# SafeTriageNet: When Getting It Wrong Matters More Than Getting It Right

*Safety-Aware Multimodal Triage with Informative Missingness and Asymmetric Clinical Cost*

**Triagegeist Competition — Laitinen-Fredriksson Foundation**

---

Emergency triage is a high-stakes ranking problem disguised as a classification problem. A
triage nurse is not only assigning a label; they are deciding who can wait and who cannot.
In that setting, all mistakes are not equal. Over-triaging a stable patient is inefficient,
but under-triaging a critically ill patient can delay life-saving care.

**SafeTriageNet** treats triage acuity prediction as a patient-safety problem, not a pure
accuracy contest. The goal is not the single most aggressive classifier on paper — it is a
model that keeps accuracy competitive while shifting the error profile away from the most
dangerous misses.

### Three pillars

1. **Informative Missingness** — missing vitals are treated as a clinical signal, not just
   as a nuisance to impute away.
2. **Multimodal feature set** — structured intake, light-weight complaint-text heuristics,
   and comorbidity history are engineered into a single modeling table.
3. **Safety-aware selection** — an asymmetric clinical cost matrix, sample weights for
   rare high-acuity classes, and entropy-triggered conservative shifting toward safer
   predictions.

This notebook is self-contained: it runs end-to-end on Kaggle without any external src/
dependency and produces `submission.csv` in the working directory.


## 1. Setup & Imports


In [ ]:
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    confusion_matrix, classification_report,
)

import lightgbm as lgb
import xgboost as xgb

SEED = 42
np.random.seed(SEED)

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.figsize": (14, 8),
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "figure.dpi": 100,
})
PALETTE = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db"]
ESI_NAMES = {1: "Resuscitation", 2: "Emergent", 3: "Urgent",
             4: "Less Urgent", 5: "Non-urgent"}

print("SafeTriageNet -- setup complete.")


## 2. Data Loading

The notebook tries the Kaggle competition data path first (`/kaggle/input/triagegeist`),
then standard Kaggle dataset mirrors, then a few local fallbacks so it can also be run
outside Kaggle. The required files are:

- `train.csv` — 80,000 training visits with the `triage_acuity` label
- `test.csv`  — 20,000 test visits
- `chief_complaints.csv` — free-text chief complaint per patient
- `patient_history.csv` — structured comorbidity flags per patient
- `sample_submission.csv` — required output format


In [ ]:
REQUIRED_FILES = [
    "train.csv",
    "test.csv",
    "chief_complaints.csv",
    "patient_history.csv",
    "sample_submission.csv",
]


def _cwd_safe() -> Path:
    try:
        return Path.cwd()
    except Exception:
        return Path(".")


def resolve_data_dir() -> Path:
    """Find the triagegeist data folder across Kaggle and local layouts."""
    candidates: List[Path] = []

    env_dir = os.environ.get("TRIAGEGEIST_DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir))

    # Kaggle competition / dataset mounts (standard Kaggle layout)
    candidates.extend([
        Path("/kaggle/input/competitions/triagegeist"),
        Path("/kaggle/input/competitions/triagegeist-data"),
        Path("/kaggle/input/competitions/triagegeist-synthetic"),
    ])

    # If a Kaggle dataset was mounted under a different name, sweep /kaggle/input
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for sub in sorted(kaggle_root.iterdir()):
            if sub.is_dir() and sub not in candidates:
                candidates.append(sub)

    # Local fallbacks
    here = _cwd_safe()
    for extra in [
        here / "triagegeist",
        here.parent / "triagegeist",
        here.parent.parent / "triagegeist",
        Path("./triagegeist"),
        Path("../triagegeist"),
        Path("../../triagegeist"),
    ]:
        candidates.append(extra)

    seen: set = set()
    for cand in candidates:
        try:
            cand = cand.resolve()
        except Exception:
            continue
        if cand in seen:
            continue
        seen.add(cand)
        if cand.is_dir() and all((cand / name).exists() for name in REQUIRED_FILES):
            return cand

    raise FileNotFoundError(
        "Could not find the triagegeist data. Expected the five CSVs under "
        "/kaggle/input/triagegeist (Kaggle) or a local ./triagegeist/ folder. "
        "Set TRIAGEGEIST_DATA_DIR to override."
    )


DATA_DIR = resolve_data_dir()
print(f"Using data directory: {DATA_DIR}")

train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
complaints = pd.read_csv(DATA_DIR / "chief_complaints.csv")
history = pd.read_csv(DATA_DIR / "patient_history.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"\n  Train set:         {train_raw.shape[0]:>7,} records x {train_raw.shape[1]} columns")
print(f"  Test set:          {test_raw.shape[0]:>7,} records x {test_raw.shape[1]} columns")
print(f"  Chief complaints:  {complaints.shape[0]:>7,} records")
print(f"  Patient history:   {history.shape[0]:>7,} records")
print(f"  Sample submission: {sample_sub.shape[0]:>7,} records")

print("\n  Target distribution (triage_acuity):")
print(train_raw["triage_acuity"].value_counts().sort_index().to_string())


## 3. Clinical Safety Module

This section defines the **asymmetric clinical cost matrix** and the clinical safety
metrics reported alongside standard ML metrics. Under-triage of ESI-1/2 patients is
penalized much more heavily than over-triage of ESI-4/5 patients.


In [ ]:
# --- Asymmetric Clinical Cost Matrix --------------------------------------------

# ESI levels: 1 (Resuscitation) -> 5 (Non-urgent)
# Matrix[actual][predicted] = clinical cost of that misclassification
# Design principles:
#   1. Under-triage (actual < predicted, i.e., more urgent than predicted) costs MORE
#   2. Cost scales super-linearly with distance from diagonal
#   3. Under-triage of ESI-1 patients is catastrophic (life-threatening delay)
#   4. Over-triage has non-zero cost (wastes scarce resuscitation resources)

CLINICAL_COST_MATRIX = np.array([
    #  Pred:  1     2     3     4     5
    [  0.0,  1.0,  4.0, 10.0, 20.0],   # Actual: 1 (Resuscitation)
    [  0.5,  0.0,  2.0,  6.0, 12.0],   # Actual: 2 (Emergent)
    [  0.3,  0.5,  0.0,  3.0,  8.0],   # Actual: 3 (Urgent)
    [  0.2,  0.3,  0.5,  0.0,  3.0],   # Actual: 4 (Less Urgent)
    [  0.1,  0.2,  0.3,  0.5,  0.0],   # Actual: 5 (Non-urgent)
], dtype=np.float64)


def get_cost_matrix() -> np.ndarray:
    """Return the asymmetric clinical cost matrix (5x5, 0-indexed)."""
    return CLINICAL_COST_MATRIX.copy()


def cost_weighted_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute the mean cost-weighted error using the clinical cost matrix.
    
    Parameters:
        y_true: Array of true ESI labels (1-5)
        y_pred: Array of predicted ESI labels (1-5)
    
    Returns:
        Mean clinical cost across all predictions. Lower is better.
    """
    costs = np.array([
        CLINICAL_COST_MATRIX[int(t) - 1, int(p) - 1]
        for t, p in zip(y_true, y_pred)
    ])
    return costs.mean()


def undertriage_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute under-triage rate: fraction of high-acuity patients (ESI 1-2)
    incorrectly predicted as lower acuity (ESI 3-5).
    
    This is the primary patient safety metric in triage evaluation.
    The American College of Surgeons recommends an under-triage rate < 5%.
    """
    high_acuity_mask = np.isin(y_true, [1, 2])
    if high_acuity_mask.sum() == 0:
        return 0.0
    
    high_acuity_true = y_true[high_acuity_mask]
    high_acuity_pred = y_pred[high_acuity_mask]
    
    undertriaged = high_acuity_pred > 2  # Predicted 3, 4, or 5
    return undertriaged.mean()


def overtriage_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute over-triage rate: fraction of low-acuity patients (ESI 4-5)
    incorrectly predicted as higher acuity (ESI 1-3).
    
    Over-triage wastes resources but doesn't directly endanger patients.
    Some over-triage is acceptable and even desirable from a safety standpoint.
    """
    low_acuity_mask = np.isin(y_true, [4, 5])
    if low_acuity_mask.sum() == 0:
        return 0.0
    
    low_acuity_true = y_true[low_acuity_mask]
    low_acuity_pred = y_pred[low_acuity_mask]
    
    overtriaged = low_acuity_pred < 4  # Predicted 1, 2, or 3
    return overtriaged.mean()


def severe_undertriage_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Severe under-triage: ESI 1-2 patients predicted as ESI 4-5.
    This is the most dangerous error class -- off by >=2 levels.
    """
    high_acuity_mask = np.isin(y_true, [1, 2])
    if high_acuity_mask.sum() == 0:
        return 0.0
    
    high_acuity_pred = y_pred[high_acuity_mask]
    severely_undertriaged = np.isin(high_acuity_pred, [4, 5])
    return severely_undertriaged.mean()


def safety_adjusted_kappa(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Weighted Cohen's Kappa using clinical cost-derived weights.
    
    Standard Cohen's Kappa uses either linear or quadratic weights.
    We use the clinical cost matrix as custom weights, so disagreements
    that are clinically dangerous contribute more to the score.
    """
    # Create a weight matrix from costs (normalized to 0-1 range)
    max_cost = CLINICAL_COST_MATRIX.max()
    weights = CLINICAL_COST_MATRIX / max_cost
    
    # Use sklearn's kappa with custom sample weights isn't directly supported,
    # so we compute via the confusion matrix
    n_classes = 5
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    
    n = cm.sum()
    if n == 0:
        return 0.0
    
    # Expected frequency under chance
    row_sums = cm.sum(axis=1)
    col_sums = cm.sum(axis=0)
    expected = np.outer(row_sums, col_sums) / n
    
    # Weighted observed and expected disagreement
    w_observed = (weights * cm).sum() / n
    w_expected = (weights * expected).sum() / n
    
    if w_expected == 0:
        return 1.0
    
    return 1.0 - (w_observed / w_expected)


def adjacent_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Fraction of predictions within +/-1 of the true ESI level.
    Clinically, being off by 1 level is often tolerable.
    """
    return (np.abs(y_true - y_pred) <= 1).mean()


# --- Comprehensive Safety Report ------------------------------------------------

def full_safety_report(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_pred_proba: Optional[np.ndarray] = None,
    model_name: str = "Model"
) -> Dict:
    """
    Generate a comprehensive evaluation report with both standard ML metrics
    and clinical safety metrics.
    
    Returns a dictionary of all metrics.
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    
    report = {
        'model_name': model_name,
        
        # Standard ML metrics
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'cohen_kappa': cohen_kappa_score(y_true, y_pred),
        'quadratic_kappa': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
        'adjacent_accuracy': adjacent_accuracy(y_true, y_pred),
        
        # Clinical safety metrics
        'undertriage_rate': undertriage_rate(y_true, y_pred),
        'severe_undertriage_rate': severe_undertriage_rate(y_true, y_pred),
        'overtriage_rate': overtriage_rate(y_true, y_pred),
        'cost_weighted_error': cost_weighted_error(y_true, y_pred),
        'safety_adjusted_kappa': safety_adjusted_kappa(y_true, y_pred),
    }
    
    # Per-class metrics
    per_class = classification_report(y_true, y_pred, labels=[1, 2, 3, 4, 5],
                                       output_dict=True, zero_division=0)
    for esi in [1, 2, 3, 4, 5]:
        key = str(esi)
        if key in per_class:
            report[f'esi{esi}_precision'] = per_class[key]['precision']
            report[f'esi{esi}_recall'] = per_class[key]['recall']
            report[f'esi{esi}_f1'] = per_class[key]['f1-score']
    
    return report


def print_safety_report(report: Dict):
    """Pretty-print a safety report."""
    print(f"\n{'='*70}")
    print(f"  EVALUATION REPORT: {report['model_name']}")
    print(f"{'='*70}")
    
    print(f"\n  -- Standard ML Metrics --")
    print(f"  Accuracy:             {report['accuracy']:.4f}")
    print(f"  Macro F1:             {report['macro_f1']:.4f}")
    print(f"  Weighted F1:          {report['weighted_f1']:.4f}")
    print(f"  Cohen's Kappa:        {report['cohen_kappa']:.4f}")
    print(f"  Quadratic Kappa:      {report['quadratic_kappa']:.4f}")
    print(f"  Adjacent Accuracy:    {report['adjacent_accuracy']:.4f}")
    
    print(f"\n  -- Clinical Safety Metrics --")
    print(f"  Under-triage Rate:    {report['undertriage_rate']:.4f}  "
          f"({'[!] HIGH' if report['undertriage_rate'] > 0.05 else '[OK] SAFE'})")
    print(f"  Severe Under-triage:  {report['severe_undertriage_rate']:.4f}  "
          f"({'[!!] CRITICAL' if report['severe_undertriage_rate'] > 0.02 else '[OK] SAFE'})")
    print(f"  Over-triage Rate:     {report['overtriage_rate']:.4f}")
    print(f"  Cost-Weighted Error:  {report['cost_weighted_error']:.4f}")
    print(f"  Safety-Adj. Kappa:    {report['safety_adjusted_kappa']:.4f}")
    
    print(f"\n  -- Per-Class Performance --")
    print(f"  {'ESI':<6} {'Precision':<12} {'Recall':<12} {'F1':<12}")
    print(f"  {'-'*42}")
    for esi in [1, 2, 3, 4, 5]:
        p = report.get(f'esi{esi}_precision', 0)
        r = report.get(f'esi{esi}_recall', 0)
        f = report.get(f'esi{esi}_f1', 0)
        label = ['Resuscitation', 'Emergent', 'Urgent', 'Less Urgent', 'Non-urgent'][esi-1]
        print(f"  ESI-{esi} ({label:<14}) {p:<12.4f} {r:<12.4f} {f:<12.4f}")
    
    print(f"\n{'='*70}\n")


# --- Custom LightGBM Loss Function ----------------------------------------------

def asymmetric_multiclass_objective(y_pred_raw: np.ndarray, dataset) -> Tuple[np.ndarray, np.ndarray]:
    """
    Custom LightGBM objective function with asymmetric clinical cost penalization.
    
    This function penalizes under-triage (predicting lower acuity than actual)
    more heavily than over-triage, using the clinical cost matrix as weights.
    
    Note: This is applied to the softmax-transformed outputs of a multiclass model.
    For LightGBM, y_pred_raw is flattened: (n_samples * n_classes,) with class-major ordering.
    """
    labels = dataset.get_label().astype(int)
    n_samples = len(labels)
    n_classes = 5
    
    # Reshape predictions to (n_samples, n_classes)
    y_pred_raw = y_pred_raw.reshape(n_classes, n_samples).T
    
    # Softmax
    exp_pred = np.exp(y_pred_raw - y_pred_raw.max(axis=1, keepdims=True))
    softmax = exp_pred / exp_pred.sum(axis=1, keepdims=True)
    
    # One-hot encode labels (0-indexed)
    labels_0idx = labels - 1  # Convert 1-5 to 0-4
    one_hot = np.zeros((n_samples, n_classes))
    one_hot[np.arange(n_samples), labels_0idx] = 1
    
    # Standard cross-entropy gradient: (softmax - one_hot)
    # We weight each class's gradient contribution by the clinical cost
    grad = softmax - one_hot
    
    # Apply asymmetric cost weighting
    for i in range(n_samples):
        true_class = labels_0idx[i]
        for c in range(n_classes):
            if c != true_class:
                cost = CLINICAL_COST_MATRIX[true_class, c]
                grad[i, c] *= (1.0 + cost)  # Scale gradient by cost
    
    # Hessian (second derivative of cross-entropy): softmax * (1 - softmax)
    hessian = softmax * (1.0 - softmax)
    
    # Apply same cost weighting to hessian
    for i in range(n_samples):
        true_class = labels_0idx[i]
        for c in range(n_classes):
            if c != true_class:
                cost = CLINICAL_COST_MATRIX[true_class, c]
                hessian[i, c] *= (1.0 + cost)
    
    # Flatten back to (n_classes * n_samples,)
    grad = grad.T.flatten()
    hessian = hessian.T.flatten()
    
    return grad, hessian


def conservative_shift(
    y_pred_proba: np.ndarray,
    entropy_threshold: float = 0.7,
    shift_strength: int = 1
) -> np.ndarray:
    """
    Uncertainty-aware conservative prediction shifting.
    
    When the model's prediction entropy exceeds a threshold (indicating uncertainty),
    the prediction is shifted toward higher acuity (lower ESI number) to err on
    the side of patient safety.
    
    Parameters:
        y_pred_proba: (n_samples, 5) array of class probabilities for ESI 1-5
        entropy_threshold: Normalized entropy threshold (0-1) above which to shift
        shift_strength: Number of ESI levels to shift toward higher acuity
    
    Returns:
        Array of adjusted predictions (1-5)
    """
    # Compute prediction entropy (normalized to 0-1)
    eps = 1e-10
    entropy = -np.sum(y_pred_proba * np.log(y_pred_proba + eps), axis=1)
    max_entropy = np.log(y_pred_proba.shape[1])
    normalized_entropy = entropy / max_entropy
    
    # Base predictions (argmax + 1 for ESI 1-5)
    predictions = y_pred_proba.argmax(axis=1) + 1
    
    # Shift uncertain predictions toward higher acuity
    uncertain_mask = normalized_entropy > entropy_threshold
    predictions[uncertain_mask] = np.maximum(
        1, predictions[uncertain_mask] - shift_strength
    )
    
    return predictions


## 4. Feature Engineering Module

Clinical-grade feature engineering: informative missingness, derived physiologic indices,
lightweight complaint-text heuristics, comorbidity burdens, and temporal context.


In [ ]:
# --- Clinical Constants ---------------------------------------------------------

# Age-adjusted normal vital sign ranges (approximate clinical reference)
VITAL_NORMS = {
    'pediatric':     {'age_range': (0, 17),  'hr': (70, 120), 'rr': (16, 24), 'sbp': (90, 120),  'temp': (36.1, 37.8)},
    'young_adult':   {'age_range': (18, 39), 'hr': (60, 100), 'rr': (12, 20), 'sbp': (100, 130), 'temp': (36.1, 37.8)},
    'middle_aged':   {'age_range': (40, 64), 'hr': (60, 100), 'rr': (12, 20), 'sbp': (100, 140), 'temp': (36.1, 37.8)},
    'elderly':       {'age_range': (65, 120),'hr': (60, 100), 'rr': (12, 20), 'sbp': (100, 150), 'temp': (36.1, 37.5)},
}

# High-acuity chief complaint keywords (from ESI triage literature)
HIGH_ACUITY_KEYWORDS = [
    'chest pain', 'shortness of breath', 'difficulty breathing', 'dyspnea',
    'altered mental', 'unresponsive', 'unconscious', 'seizure', 'stroke',
    'cardiac arrest', 'anaphylaxis', 'choking', 'apnea', 'cyanosis',
    'hematemesis', 'massive bleeding', 'hemorrhage', 'trauma', 'intubat',
    'overdose', 'poisoning', 'syncope', 'collapse', 'face droop',
    'slurred speech', 'focal weakness', 'thunderclap headache',
    'suicidal', 'self-harm', 'unstable', 'pulseless',
    'respiratory distress', 'respiratory failure', 'status epilepticus',
    'acute abdomen', 'ruptured', 'dissection', 'testicular torsion',
    'ectopic pregnancy', 'meningitis', 'sepsis', 'wound', 'laceration',
]

MODERATE_ACUITY_KEYWORDS = [
    'abdominal pain', 'back pain', 'fever', 'vomiting', 'diarrhea',
    'headache', 'dizziness', 'fall', 'fracture', 'burn',
    'allergic reaction', 'asthma', 'wheezing', 'palpitation',
    'nosebleed', 'epistaxis', 'cellulitis', 'abscess', 'urinary',
    'dysuria', 'hematuria', 'flank pain', 'kidney stone',
    'swelling', 'dehydration', 'diabetic', 'hyperglycemia',
]

LOW_ACUITY_KEYWORDS = [
    'rash', 'itch', 'insect bite', 'sprain', 'strain', 'contusion',
    'sore throat', 'cold symptoms', 'cough', 'earache', 'toothache',
    'prescription refill', 'medication refill', 'follow-up',
    'suture removal', 'wound check', 'discharge instructions',
    'contraception', 'general health', 'information', 'advice',
]


# --- Missingness Features -------------------------------------------------------

VITAL_COLUMNS = [
    'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
    'pulse_pressure', 'heart_rate', 'respiratory_rate',
    'temperature_c', 'spo2', 'gcs_total', 'pain_score',
    'weight_kg', 'height_cm', 'bmi', 'shock_index', 'news2_score'
]

# Core vitals that are most informative when missing
CORE_VITALS = [
    'systolic_bp', 'diastolic_bp', 'heart_rate',
    'respiratory_rate', 'temperature_c', 'spo2'
]

# Categorical columns expected in the raw intake feed
CATEGORY_COLUMNS = [
    'arrival_mode', 'arrival_day', 'arrival_season', 'shift',
    'age_group', 'sex', 'language', 'insurance_type',
    'transport_origin', 'pain_location', 'mental_status_triage',
    'chief_complaint_system'
]


def create_missingness_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create informative missingness features.
    
    The absence of vital sign documentation in the ED is clinically meaningful:
    - Low-acuity patients often have fewer vitals recorded
    - High-acuity patients in extremis may also have missing vitals (staff focused on resuscitation)
    - The *pattern* of which vitals are missing encodes triage nurse clinical judgment
    
    Returns DataFrame with new missingness feature columns appended.
    """
    df = df.copy()
    
    # 1. Binary missingness indicators for each vital
    for col in VITAL_COLUMNS:
        if col in df.columns:
            # Handle both NaN and -1 encoding
            if col == 'pain_score':
                df[f'{col}_missing'] = (df[col].isna() | (df[col] == -1)).astype(int)
            else:
                df[f'{col}_missing'] = df[col].isna().astype(int)
    
    # 2. Documentation Completeness Score (fraction of core vitals recorded)
    missing_cols = [f'{c}_missing' for c in CORE_VITALS if f'{c}_missing' in df.columns]
    if missing_cols:
        df['doc_completeness'] = 1.0 - df[missing_cols].mean(axis=1)
    
    # 3. Total missing vitals count
    all_missing_cols = [c for c in df.columns if c.endswith('_missing')]
    if all_missing_cols:
        df['n_missing_vitals'] = df[all_missing_cols].sum(axis=1)
    
    # 4. Missingness pattern hash -- each unique pattern of missing/present is a categorical feature
    if missing_cols:
        df['missingness_pattern'] = df[missing_cols].astype(str).agg(''.join, axis=1)
    
    return df


# --- Derived Clinical Features --------------------------------------------------

def create_clinical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive clinically meaningful physiological indices from raw vitals.
    
    These composite scores are standard clinical decision tools that compress
    multiple vital signs into single risk indicators. They are well-validated
    in the emergency medicine literature.
    """
    df = df.copy()
    
    # --- Shock Index (SI) ---
    # SI = HR / SBP. Normal: 0.5-0.7. Elevated (>0.9) strongly predicts hemodynamic instability.
    # Already provided in dataset but we recompute for robustness
    if 'heart_rate' in df.columns and 'systolic_bp' in df.columns:
        mask = (df['systolic_bp'].notna()) & (df['systolic_bp'] > 0) & (df['heart_rate'].notna())
        df['shock_index_calc'] = np.nan
        df.loc[mask, 'shock_index_calc'] = df.loc[mask, 'heart_rate'] / df.loc[mask, 'systolic_bp']
        df['shock_index_elevated'] = (df['shock_index_calc'] > 0.9).astype(float)
        df['shock_index_critical'] = (df['shock_index_calc'] > 1.3).astype(float)
    
    # --- Mean Arterial Pressure (MAP) ---
    # MAP = DBP + (SBP - DBP) / 3. Critical threshold: MAP < 65 mmHg
    if 'systolic_bp' in df.columns and 'diastolic_bp' in df.columns:
        mask = df['systolic_bp'].notna() & df['diastolic_bp'].notna()
        df['map_calc'] = np.nan
        df.loc[mask, 'map_calc'] = (
            df.loc[mask, 'diastolic_bp'] + 
            (df.loc[mask, 'systolic_bp'] - df.loc[mask, 'diastolic_bp']) / 3
        )
        df['map_critical'] = (df['map_calc'] < 65).astype(float)
    
    # --- Pulse Pressure ---
    # PP = SBP - DBP. Narrow PP (<25) -> cardiogenic shock. Wide PP (>60) -> sepsis, aortic regurgitation
    if 'systolic_bp' in df.columns and 'diastolic_bp' in df.columns:
        mask = df['systolic_bp'].notna() & df['diastolic_bp'].notna()
        df['pulse_pressure_calc'] = np.nan
        df.loc[mask, 'pulse_pressure_calc'] = df.loc[mask, 'systolic_bp'] - df.loc[mask, 'diastolic_bp']
        df['pp_narrow'] = (df['pulse_pressure_calc'] < 25).astype(float)
        df['pp_wide'] = (df['pulse_pressure_calc'] > 60).astype(float)
    
    # --- SpO2 severity tiers ---
    if 'spo2' in df.columns:
        df['spo2_critical'] = (df['spo2'] < 90).astype(float)
        df['spo2_concerning'] = ((df['spo2'] >= 90) & (df['spo2'] < 94)).astype(float)
        df['spo2_normal'] = (df['spo2'] >= 94).astype(float)
    
    # --- GCS severity tiers ---
    if 'gcs_total' in df.columns:
        df['gcs_severe'] = (df['gcs_total'] <= 8).astype(float)
        df['gcs_moderate'] = ((df['gcs_total'] > 8) & (df['gcs_total'] <= 12)).astype(float)
        df['gcs_mild'] = (df['gcs_total'] == 15).astype(float)  # Normal
    
    # --- Temperature severity ---
    if 'temperature_c' in df.columns:
        df['hypothermia'] = (df['temperature_c'] < 35.5).astype(float)
        df['fever'] = (df['temperature_c'] >= 38.0).astype(float)
        df['high_fever'] = (df['temperature_c'] >= 39.0).astype(float)
        df['hyperthermia'] = (df['temperature_c'] >= 40.0).astype(float)
    
    # --- Heart rate severity ---
    if 'heart_rate' in df.columns:
        df['bradycardia'] = (df['heart_rate'] < 50).astype(float)
        df['tachycardia'] = (df['heart_rate'] > 100).astype(float)
        df['severe_tachycardia'] = (df['heart_rate'] > 130).astype(float)
    
    # --- Respiratory rate severity ---
    if 'respiratory_rate' in df.columns:
        df['tachypnea'] = (df['respiratory_rate'] > 22).astype(float)
        df['severe_tachypnea'] = (df['respiratory_rate'] > 30).astype(float)
        df['bradypnea'] = (df['respiratory_rate'] < 10).astype(float)
    
    # --- Blood pressure severity ---
    if 'systolic_bp' in df.columns:
        df['hypotension'] = (df['systolic_bp'] < 90).astype(float)
        df['severe_hypotension'] = (df['systolic_bp'] < 70).astype(float)
        df['hypertensive_urgency'] = (df['systolic_bp'] > 180).astype(float)
    
    # --- Pain severity tiers ---
    if 'pain_score' in df.columns:
        df['pain_none'] = (df['pain_score'] == 0).astype(float)
        df['pain_mild'] = ((df['pain_score'] >= 1) & (df['pain_score'] <= 3)).astype(float)
        df['pain_moderate'] = ((df['pain_score'] >= 4) & (df['pain_score'] <= 6)).astype(float)
        df['pain_severe'] = ((df['pain_score'] >= 7) & (df['pain_score'] <= 10)).astype(float)
    
    # --- Age-adjusted vital sign abnormality flags ---
    if 'age' in df.columns and 'heart_rate' in df.columns:
        df['hr_age_abnormal'] = 0.0
        for group, norms in VITAL_NORMS.items():
            lo, hi = norms['age_range']
            hr_lo, hr_hi = norms['hr']
            mask = (df['age'] >= lo) & (df['age'] <= hi) & df['heart_rate'].notna()
            df.loc[mask & ((df['heart_rate'] < hr_lo) | (df['heart_rate'] > hr_hi)), 'hr_age_abnormal'] = 1.0
    
    # --- Composite critical count ---
    # How many critical flags does this patient have? Strong ordinal signal.
    critical_flags = [c for c in df.columns if c in [
        'shock_index_critical', 'map_critical', 'spo2_critical',
        'gcs_severe', 'hypothermia', 'hyperthermia', 'severe_tachycardia',
        'severe_tachypnea', 'severe_hypotension', 'bradypnea'
    ]]
    if critical_flags:
        df['n_critical_flags'] = df[critical_flags].sum(axis=1)
    
    return df


# --- Temporal Features -----------------------------------------------------------

def create_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Encode temporal patterns from arrival time and day.
    
    ED volumes, case mix, and staffing patterns vary systematically by time of day
    and day of week. Night and weekend shifts are associated with higher acuity
    casemix and potentially higher triage error rates.
    """
    df = df.copy()
    
    # Cyclical encoding of hour (captures the circular nature of time)
    if 'arrival_hour' in df.columns:
        df['hour_sin'] = np.sin(2 * np.pi * df['arrival_hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['arrival_hour'] / 24)
        
        # Clinical shift categories
        df['is_night_shift'] = ((df['arrival_hour'] >= 23) | (df['arrival_hour'] < 7)).astype(int)
        df['is_evening_shift'] = ((df['arrival_hour'] >= 15) & (df['arrival_hour'] < 23)).astype(int)
    
    # Day of week encoding
    if 'arrival_day' in df.columns:
        day_map = {
            'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
            'Friday': 4, 'Saturday': 5, 'Sunday': 6
        }
        df['day_numeric'] = df['arrival_day'].map(day_map)
        df['day_sin'] = np.sin(2 * np.pi * df['day_numeric'] / 7)
        df['day_cos'] = np.cos(2 * np.pi * df['day_numeric'] / 7)
        df['is_weekend'] = (df['arrival_day'].isin(['Saturday', 'Sunday'])).astype(int)
    
    # Arrival month cyclical
    if 'arrival_month' in df.columns:
        df['month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12)
    
    return df


# --- NLP Features ----------------------------------------------------------------

def create_nlp_features(df: pd.DataFrame, complaint_col: str = 'chief_complaint_raw') -> pd.DataFrame:
    """
    Extract clinically meaningful features from free-text chief complaint narratives.
    
    Uses keyword-based clinical classification rather than heavy NLP models,
    making the approach interpretable and reproducible without GPU requirements.
    """
    df = df.copy()
    
    if complaint_col not in df.columns:
        return df
    
    text = df[complaint_col].fillna('').str.lower()
    
    # Complaint length (very short complaints from unconscious/critical patients)
    df['complaint_length'] = text.str.len()
    df['complaint_word_count'] = text.str.split().str.len().fillna(0)
    
    # High-acuity keyword match count  
    df['n_high_acuity_keywords'] = 0
    for keyword in HIGH_ACUITY_KEYWORDS:
        df['n_high_acuity_keywords'] += text.str.contains(keyword, na=False).astype(int)
    
    # Moderate-acuity keyword match count
    df['n_moderate_acuity_keywords'] = 0
    for keyword in MODERATE_ACUITY_KEYWORDS:
        df['n_moderate_acuity_keywords'] += text.str.contains(keyword, na=False).astype(int)
    
    # Low-acuity keyword match count
    df['n_low_acuity_keywords'] = 0
    for keyword in LOW_ACUITY_KEYWORDS:
        df['n_low_acuity_keywords'] += text.str.contains(keyword, na=False).astype(int)
    
    # Acuity keyword balance
    df['keyword_acuity_signal'] = df['n_high_acuity_keywords'] - df['n_low_acuity_keywords']
    
    # Specific high-risk flags
    df['cc_chest_pain'] = text.str.contains('chest pain|chest tightness|angina', na=False).astype(int)
    df['cc_sob'] = text.str.contains('shortness of breath|difficulty breathing|dyspnea|breathless', na=False).astype(int)
    df['cc_ams'] = text.str.contains('altered mental|confusion|confused|disoriented|unresponsive|unconscious', na=False).astype(int)
    df['cc_trauma'] = text.str.contains('trauma|mva|motor vehicle|assault|fall from|stabbing|gunshot|laceration|wound', na=False).astype(int)
    df['cc_neuro'] = text.str.contains('seizure|stroke|weakness|numbness|paralysis|face droop|slurred|aphasia|thunderclap', na=False).astype(int)
    df['cc_cardiac'] = text.str.contains('cardiac|heart attack|palpitation|arrhythmia|chest pain', na=False).astype(int)
    df['cc_gi_bleed'] = text.str.contains('blood in stool|hematemesis|melena|rectal bleeding|vomiting blood', na=False).astype(int)
    df['cc_mental_health'] = text.str.contains('suicid|self-harm|overdose|psychiatric|psychosis', na=False).astype(int)
    df['cc_respiratory'] = text.str.contains('asthma|wheezing|copd|respiratory|croup|stridor', na=False).astype(int)
    df['cc_sepsis'] = text.str.contains('sepsis|septic|infection.*fever|fever.*infection', na=False).astype(int)
    df['cc_abdominal'] = text.str.contains('abdominal pain|stomach pain|belly pain|acute abdomen', na=False).astype(int)
    
    # Urgency modifiers
    df['cc_acute'] = text.str.contains('acute|sudden|abrupt|worst|severe|excruciating|unbearable|worsening', na=False).astype(int)
    df['cc_chronic'] = text.str.contains('chronic|ongoing|recurrent|intermittent|long-standing|months|years', na=False).astype(int)
    
    return df


# --- Comorbidity Features --------------------------------------------------------

def create_comorbidity_features(df: pd.DataFrame, history_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """
    Aggregate patient history flags into clinically meaningful risk composites.
    """
    df = df.copy()
    
    if history_df is not None:
        # Merge patient history
        hx_cols = [c for c in history_df.columns if c.startswith('hx_')]
        df = df.merge(history_df[['patient_id'] + hx_cols], on='patient_id', how='left')
    
    hx_cols = [c for c in df.columns if c.startswith('hx_')]
    
    if hx_cols:
        # Total comorbidity burden
        df['total_hx_flags'] = df[hx_cols].sum(axis=1)
        
        # Cardiovascular risk cluster
        cv_cols = [c for c in hx_cols if c in [
            'hx_hypertension', 'hx_heart_failure', 'hx_atrial_fibrillation',
            'hx_coronary_artery_disease', 'hx_peripheral_vascular_disease', 'hx_stroke_prior'
        ]]
        if cv_cols:
            df['hx_cardiovascular_burden'] = df[cv_cols].sum(axis=1)
        
        # Metabolic risk cluster
        met_cols = [c for c in hx_cols if c in [
            'hx_diabetes_type1', 'hx_diabetes_type2', 'hx_obesity',
            'hx_hypothyroidism', 'hx_hyperthyroidism'
        ]]
        if met_cols:
            df['hx_metabolic_burden'] = df[met_cols].sum(axis=1)
        
        # Immunocompromised risk
        immune_cols = [c for c in hx_cols if c in [
            'hx_immunosuppressed', 'hx_hiv', 'hx_malignancy'
        ]]
        if immune_cols:
            df['hx_immunocompromised'] = (df[immune_cols].sum(axis=1) > 0).astype(int)
        
        # Respiratory risk
        resp_cols = [c for c in hx_cols if c in ['hx_asthma', 'hx_copd']]
        if resp_cols:
            df['hx_respiratory_risk'] = df[resp_cols].sum(axis=1)
        
        # Mental health risk
        mh_cols = [c for c in hx_cols if c in [
            'hx_depression', 'hx_anxiety', 'hx_substance_use_disorder'
        ]]
        if mh_cols:
            df['hx_mental_health_burden'] = df[mh_cols].sum(axis=1)
        
        # Bleeding risk
        if 'hx_coagulopathy' in df.columns:
            df['hx_bleeding_risk'] = df['hx_coagulopathy']
        
        # High complexity flag (>= 5 comorbidities)
        df['hx_high_complexity'] = (df['total_hx_flags'] >= 5).astype(int)
    
    return df


# --- Master Pipeline ------------------------------------------------------------

def engineer_features(
    df: pd.DataFrame,
    complaints_df: Optional[pd.DataFrame] = None,
    history_df: Optional[pd.DataFrame] = None,
    is_train: bool = True
) -> pd.DataFrame:
    """
    Master feature engineering pipeline. Applies all feature transforms in order.
    
    Parameters:
        df: Main patient intake DataFrame (train.csv or test.csv)
        complaints_df: Chief complaints DataFrame
        history_df: Patient history DataFrame  
        is_train: Whether this is training data (controls target handling)
    
    Returns:
        DataFrame with all engineered features
    """
    print(f"[Features] Starting feature engineering on {len(df)} records...")
    
    # Merge chief complaints if provided
    if complaints_df is not None:
        df = df.merge(complaints_df[['patient_id', 'chief_complaint_raw']], 
                       on='patient_id', how='left')
        print(f"  -> Merged chief complaints")
    
    # Handle pain_score -1 encoding -> NaN
    if 'pain_score' in df.columns:
        df['pain_score_reported'] = (df['pain_score'] != -1).astype(int)
        df.loc[df['pain_score'] == -1, 'pain_score'] = np.nan
    
    # Apply feature pipelines
    df = create_missingness_features(df)
    print(f"  -> Missingness features created")
    
    df = create_clinical_features(df)
    print(f"  -> Clinical features created")
    
    df = create_temporal_features(df)
    print(f"  -> Temporal features created")
    
    df = create_nlp_features(df)
    print(f"  -> NLP features created")
    
    df = create_comorbidity_features(df, history_df)
    print(f"  -> Comorbidity features created")
    
    total_features = len([c for c in df.columns if c not in ['patient_id', 'triage_acuity']])
    print(f"[Features] Complete. Total features: {total_features}")
    
    return df


def get_feature_columns(df: pd.DataFrame) -> List[str]:
    """
    Return the list of feature columns suitable for modeling.
    Excludes IDs, targets, raw text, and intermediate columns.
    """
    exclude = {
        'patient_id', 'triage_acuity', 'chief_complaint_raw',
        'missingness_pattern', 'chief_complaint_system',
        'disposition', 'ed_los_hours',  # leakage: these are post-triage outcomes
        'site_id', 'triage_nurse_id',   # bias: institutional/nurse-specific patterns
    }
    
    # Get all numeric and boolean columns
    feature_cols = []
    for col in df.columns:
        if col in exclude:
            continue
        if df[col].dtype in ['float64', 'float32', 'int64', 'int32', 'uint8', 'bool']:
            feature_cols.append(col)
    
    return feature_cols


def build_category_maps(df: pd.DataFrame) -> Dict[str, List]:
    """
    Fit categorical vocabularies on a reference DataFrame.

    These mappings should be learned on the training split and then reused for
    validation/test data so encoded category IDs remain stable.
    """
    category_maps: Dict[str, List] = {}

    for col in CATEGORY_COLUMNS:
        if col in df.columns:
            values = pd.Series(df[col]).dropna().unique().tolist()
            category_maps[col] = values

    return category_maps


def encode_categoricals(
    df: pd.DataFrame,
    category_maps: Optional[Dict[str, List]] = None
) -> pd.DataFrame:
    """
    Encode categorical columns for tree-based models.

    When ``category_maps`` is provided, categories are encoded against that
    fixed vocabulary. Unseen values are mapped to ``-1``.
    """
    df = df.copy()

    if category_maps is None:
        category_maps = build_category_maps(df)

    for col in CATEGORY_COLUMNS:
        if col in df.columns:
            categories = category_maps.get(col)
            if categories is None:
                categories = pd.Series(df[col]).dropna().unique().tolist()
            df[col] = pd.Categorical(df[col], categories=categories).codes.astype(np.int32)

    return df


## 5. Modeling Module

5-fold stratified LightGBM and XGBoost base learners, stacking meta-learner with its own
out-of-fold evaluation, and the prediction pipeline that feeds test data through the
full stack.


In [ ]:
# --- LightGBM Configuration ----------------------------------------------------

LGBM_BASE_PARAMS = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'max_depth': 8,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbose': -1,
    'seed': 42,
    'n_jobs': -1,
}

# Class weights to address class imbalance (ESI-1 is rare, ESI-3 is dominant)
# These give MORE weight to rare, high-acuity classes
ACUITY_SAMPLE_WEIGHTS = {
    1: 5.0,   # Resuscitation -- rare, must not miss
    2: 3.0,   # Emergent
    3: 1.0,   # Urgent -- most common
    4: 1.5,   # Less urgent
    5: 2.0,   # Non-urgent -- also needs correct identification
}

XGB_BASE_PARAMS = {
    'objective': 'multi:softprob',
    'num_class': 5,
    'eval_metric': 'mlogloss',
    'max_depth': 7,
    'learning_rate': 0.05,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'min_child_weight': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'tree_method': 'hist',
    'seed': 42,
    'verbosity': 0,
}


# --- Training Utilities --------------------------------------------------------

def create_sample_weights(y: np.ndarray) -> np.ndarray:
    """Create sample weights from class-aware weighting scheme."""
    return np.array([ACUITY_SAMPLE_WEIGHTS.get(label, 1.0) for label in y])


def _build_meta_model() -> LogisticRegression:
    """Create a fresh stacking meta-learner instance."""
    return LogisticRegression(
        C=1.0,
        max_iter=1000,
        multi_class='multinomial',
        solver='lbfgs',
        random_state=42,
    )


def train_lgbm_cv(
    X: pd.DataFrame,
    y: np.ndarray,
    feature_cols: List[str],
    n_folds: int = 5,
    n_boost_rounds: int = 1500,
    early_stopping_rounds: int = 50,
    params: Optional[Dict] = None,
    use_sample_weights: bool = True,
    model_name: str = "LightGBM"
) -> Tuple[List, np.ndarray, np.ndarray]:
    """
    Train LightGBM with stratified K-fold cross-validation.
    
    Returns:
        models: List of trained LightGBM Booster models
        oof_preds: Out-of-fold probability predictions (n_samples, 5)
        oof_labels: Out-of-fold predicted labels (n_samples,)
    """
    if params is None:
        params = LGBM_BASE_PARAMS.copy()
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    oof_preds = np.zeros((len(X), 5))
    models = []
    
    print(f"\n{'-'*60}")
    print(f"  Training {model_name} ({n_folds}-fold CV)")
    print(f"{'-'*60}")
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train = X.iloc[train_idx][feature_cols]
        y_train = y[train_idx]
        X_val = X.iloc[val_idx][feature_cols]
        y_val = y[val_idx]
        
        # Convert labels from 1-5 to 0-4 for LightGBM
        y_train_0idx = y_train - 1
        y_val_0idx = y_val - 1
        
        # Create sample weights
        w_train = create_sample_weights(y_train) if use_sample_weights else None
        
        dtrain = lgb.Dataset(X_train, label=y_train_0idx, weight=w_train)
        dval = lgb.Dataset(X_val, label=y_val_0idx, reference=dtrain)
        
        callbacks = [
            lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ]
        
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=n_boost_rounds,
            valid_sets=[dval],
            callbacks=callbacks,
        )
        
        val_preds = model.predict(X_val)
        oof_preds[val_idx] = val_preds
        models.append(model)
        
        val_labels = val_preds.argmax(axis=1) + 1
        fold_acc = (val_labels == y_val).mean()
        print(f"  Fold {fold_idx + 1}/{n_folds}  |  "
              f"Best iteration: {model.best_iteration}  |  "
              f"Val Accuracy: {fold_acc:.4f}")
    
    oof_labels = oof_preds.argmax(axis=1) + 1
    overall_acc = (oof_labels == y).mean()
    print(f"\n  -> Overall OOF Accuracy: {overall_acc:.4f}")
    
    return models, oof_preds, oof_labels


def train_xgb_cv(
    X: pd.DataFrame,
    y: np.ndarray,
    feature_cols: List[str],
    n_folds: int = 5,
    n_boost_rounds: int = 1500,
    early_stopping_rounds: int = 50,
    params: Optional[Dict] = None,
    use_sample_weights: bool = True,
    model_name: str = "XGBoost"
) -> Tuple[List, np.ndarray, np.ndarray]:
    """
    Train XGBoost with stratified K-fold cross-validation.
    """
    if params is None:
        params = XGB_BASE_PARAMS.copy()
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    oof_preds = np.zeros((len(X), 5))
    models = []
    
    print(f"\n{'-'*60}")
    print(f"  Training {model_name} ({n_folds}-fold CV)")
    print(f"{'-'*60}")
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train = X.iloc[train_idx][feature_cols].values
        y_train = y[train_idx]
        X_val = X.iloc[val_idx][feature_cols].values
        y_val = y[val_idx]
        
        # Convert labels from 1-5 to 0-4
        y_train_0idx = y_train - 1
        y_val_0idx = y_val - 1
        
        w_train = create_sample_weights(y_train) if use_sample_weights else None
        
        dtrain = xgb.DMatrix(X_train, label=y_train_0idx, weight=w_train)
        dval = xgb.DMatrix(X_val, label=y_val_0idx)
        
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=n_boost_rounds,
            evals=[(dval, 'val')],
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False,
        )
        
        val_preds = model.predict(dval)
        oof_preds[val_idx] = val_preds
        models.append(model)
        
        val_labels = val_preds.argmax(axis=1) + 1
        fold_acc = (val_labels == y_val).mean()
        print(f"  Fold {fold_idx + 1}/{n_folds}  |  "
              f"Best iteration: {model.best_iteration}  |  "
              f"Val Accuracy: {fold_acc:.4f}")
    
    oof_labels = oof_preds.argmax(axis=1) + 1
    overall_acc = (oof_labels == y).mean()
    print(f"\n  -> Overall OOF Accuracy: {overall_acc:.4f}")
    
    return models, oof_preds, oof_labels


# --- Stacking Meta-Learner -----------------------------------------------------

def train_stacking_meta(
    oof_predictions: Dict[str, np.ndarray],
    y_true: np.ndarray,
    n_folds: int = 5,
) -> Tuple[LogisticRegression, StandardScaler, np.ndarray, np.ndarray]:
    """
    Train a stacking meta-learner on out-of-fold predictions from base models.
    
    The meta-learner is evaluated with its own stratified cross-validation so the
    reported ensemble metrics are based on honest out-of-fold predictions rather
    than the same meta-features used for fitting.
    
    Parameters:
        oof_predictions: Dict of model_name -> (n_samples, 5) OOF probability arrays
        y_true: True labels (1-5)
        n_folds: Number of CV folds for meta-learner training
    
    Returns:
        Fitted full-data LogisticRegression model and StandardScaler, plus
        meta-learner OOF probabilities and OOF labels for evaluation
    """
    meta_features = np.hstack(list(oof_predictions.values()))
    classes = np.sort(np.unique(y_true))
    class_to_idx = {label: idx for idx, label in enumerate(classes)}
    sample_weights = create_sample_weights(y_true)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    oof_meta_proba = np.zeros((len(y_true), len(classes)))
    oof_meta_labels = np.zeros(len(y_true), dtype=int)
    
    print(f"\n{'-'*60}")
    print(f"  Training Stacking Meta-Learner")
    print(f"  Input models: {list(oof_predictions.keys())}")
    print(f"  Meta-feature dimension: {meta_features.shape[1]}")
    print(f"{'-'*60}")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(meta_features, y_true)):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(meta_features[train_idx])
        X_val = scaler.transform(meta_features[val_idx])

        meta_model = _build_meta_model()
        meta_model.fit(
            X_train,
            y_true[train_idx],
            sample_weight=sample_weights[train_idx]
        )

        val_proba = meta_model.predict_proba(X_val)
        val_labels = meta_model.predict(X_val)

        for local_idx, class_label in enumerate(meta_model.classes_):
            oof_meta_proba[val_idx, class_to_idx[class_label]] = val_proba[:, local_idx]
        oof_meta_labels[val_idx] = val_labels

        fold_acc = (val_labels == y_true[val_idx]).mean()
        print(f"  Fold {fold_idx + 1}/{n_folds}  |  Val Accuracy: {fold_acc:.4f}")

    meta_oof_acc = (oof_meta_labels == y_true).mean()
    print(f"\n  -> Meta-learner OOF Accuracy: {meta_oof_acc:.4f}")

    full_scaler = StandardScaler()
    meta_features_scaled = full_scaler.fit_transform(meta_features)

    full_meta_model = _build_meta_model()
    full_meta_model.fit(
        meta_features_scaled,
        y_true,
        sample_weight=sample_weights
    )

    return full_meta_model, full_scaler, oof_meta_proba, oof_meta_labels


def predict_stacked(
    models_dict: Dict[str, List],
    X_test: pd.DataFrame,
    feature_cols: List[str],
    meta_model: LogisticRegression,
    scaler: StandardScaler,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate predictions from the full stacking pipeline.
    
    Parameters:
        models_dict: Dict of model_name -> list of fold models
        X_test: Test features DataFrame  
        feature_cols: Feature column names
        meta_model: Fitted meta-learner
        scaler: Fitted StandardScaler
    
    Returns:
        Probability predictions (n_samples, 5) and label predictions (n_samples,)
    """
    test_preds_list = []
    
    for model_name, fold_models in models_dict.items():
        model_preds = np.zeros((len(X_test), 5))
        
        for model in fold_models:
            if isinstance(model, lgb.Booster):
                preds = model.predict(X_test[feature_cols])
            elif isinstance(model, xgb.Booster):
                dtest = xgb.DMatrix(X_test[feature_cols].values)
                preds = model.predict(dtest)
            else:
                raise ValueError(f"Unknown model type: {type(model)}")
            
            model_preds += preds
        
        model_preds /= len(fold_models)
        test_preds_list.append(model_preds)
    
    # Stack and transform through meta-learner
    meta_features = np.hstack(test_preds_list)
    meta_features_scaled = scaler.transform(meta_features)
    
    proba = meta_model.predict_proba(meta_features_scaled)
    labels = meta_model.predict(meta_features_scaled)
    
    return proba, labels


# --- Feature Importance --------------------------------------------------------

def get_feature_importance(
    models: List,
    feature_cols: List[str],
    importance_type: str = 'gain'
) -> pd.DataFrame:
    """
    Aggregate feature importance across CV fold models.
    """
    importance_df = pd.DataFrame({'feature': feature_cols})
    
    for i, model in enumerate(models):
        if isinstance(model, lgb.Booster):
            imp = model.feature_importance(importance_type=importance_type)
        else:
            continue
        importance_df[f'fold_{i}'] = imp
    
    fold_cols = [c for c in importance_df.columns if c.startswith('fold_')]
    importance_df['mean_importance'] = importance_df[fold_cols].mean(axis=1)
    importance_df['std_importance'] = importance_df[fold_cols].std(axis=1)
    importance_df = importance_df.sort_values('mean_importance', ascending=False)
    
    return importance_df[['feature', 'mean_importance', 'std_importance']]


## 6. Exploratory Data Analysis

Two questions drive the EDA:

1. **Is the class imbalance big enough to matter for evaluation?** Yes — ESI-1 is the
   rarest class, and a single aggregate accuracy score hides what happens on the patients
   who need triage support the most.
2. **Is missingness informative?** The dataset description claims so. We test it with a
   chi-square association test per vital.

All figures in this section are saved alongside the notebook.


In [ ]:
OUTPUT_DIR = Path(os.environ.get("SAFETRIAGE_OUTPUT_DIR", "."))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -- 6a: Target distribution + cost matrix --
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
counts = train_raw["triage_acuity"].value_counts().sort_index()
colors = [PALETTE[i] for i in range(5)]
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_xlabel("ESI Acuity Level", fontweight="bold")
axes[0].set_ylabel("Patient Count", fontweight="bold")
axes[0].set_title("Triage Acuity Distribution (Training Set)")
for bar, count in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 200,
        f"{count:,}\n({count / len(train_raw) * 100:.1f}%)",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
    )
for esi, name in ESI_NAMES.items():
    axes[0].text(esi, -max(counts.values) * 0.06, name,
                 ha="center", fontsize=8, style="italic", color="gray")

im = axes[1].imshow(CLINICAL_COST_MATRIX, cmap="YlOrRd", aspect="auto")
axes[1].set_xticks(range(5))
axes[1].set_yticks(range(5))
axes[1].set_xticklabels([f"ESI-{i + 1}" for i in range(5)])
axes[1].set_yticklabels([f"ESI-{i + 1}" for i in range(5)])
axes[1].set_xlabel("Predicted Acuity", fontweight="bold")
axes[1].set_ylabel("Actual Acuity", fontweight="bold")
axes[1].set_title("Asymmetric Clinical Cost Matrix")
for i in range(5):
    for j in range(5):
        color = "white" if CLINICAL_COST_MATRIX[i, j] > 8 else "black"
        axes[1].text(j, i, f"{CLINICAL_COST_MATRIX[i, j]:.1f}",
                     ha="center", va="center", fontsize=12, fontweight="bold", color=color)
plt.colorbar(im, ax=axes[1], label="Clinical Cost")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig1_target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -- 6b: Informative missingness --
print("\n-- Informative missingness analysis --")
vital_cols_present = [c for c in CORE_VITALS if c in train_raw.columns]
rows = []
for col in vital_cols_present:
    for acuity in sorted(train_raw["triage_acuity"].unique()):
        subset = train_raw[train_raw["triage_acuity"] == acuity]
        rows.append({"vital": col, "acuity": acuity,
                     "missing_rate": subset[col].isna().mean()})
if "pain_score" in train_raw.columns:
    for acuity in sorted(train_raw["triage_acuity"].unique()):
        subset = train_raw[train_raw["triage_acuity"] == acuity]
        rows.append({"vital": "pain_score", "acuity": acuity,
                     "missing_rate": (subset["pain_score"] == -1).mean()})

miss_by_acuity = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
pivot = miss_by_acuity.pivot(index="vital", columns="acuity", values="missing_rate")
sns.heatmap(
    pivot, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=axes[0],
    linewidths=1, linecolor="white",
    xticklabels=[f"ESI-{i}" for i in pivot.columns],
    cbar_kws={"label": "Missing Rate"},
)
axes[0].set_title("Vital Sign Missing Rate by ESI Acuity Level\n(Higher = More Missing)",
                  fontweight="bold")
axes[0].set_xlabel("Triage Acuity (ESI)", fontweight="bold")
axes[0].set_ylabel("")

n_vitals_check = train_raw[vital_cols_present].notna().sum(axis=1)
doc_tmp = n_vitals_check / len(vital_cols_present)
doc_comp_by_acuity = doc_tmp.groupby(train_raw["triage_acuity"]).mean()
bars = axes[1].bar(doc_comp_by_acuity.index, doc_comp_by_acuity.values,
                   color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_xlabel("ESI Acuity Level", fontweight="bold")
axes[1].set_ylabel("Mean Documentation Completeness", fontweight="bold")
axes[1].set_title("Vital Sign Documentation Completeness by Acuity\n(Higher = More Vitals Recorded)",
                  fontweight="bold")
axes[1].set_ylim(0, 1.05)
for bar, val in zip(bars, doc_comp_by_acuity.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2.0,
                 bar.get_height() + 0.01, f"{val:.3f}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_missingness_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nChi-square test: missingness x acuity")
significant, non_significant = [], []
for col in vital_cols_present:
    missing_flag = train_raw[col].isna().astype(int)
    contingency = pd.crosstab(missing_flag, train_raw["triage_acuity"])
    chi2, p_val, _, _ = stats.chi2_contingency(contingency)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(f"  {col:<22} chi2={chi2:>10.1f}  p={p_val:.2e}  {sig}")
    (significant if p_val < 0.05 else non_significant).append(col)
print(f"\n  -> {len(significant)}/{len(vital_cols_present)} core vitals show significant acuity-linked missingness.")
if non_significant:
    print(f"     Non-significant: {', '.join(non_significant)}")


In [ ]:
# -- 6c: Vital distributions by acuity --
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
vital_plot_cols = ["heart_rate", "systolic_bp", "respiratory_rate",
                   "spo2", "temperature_c", "gcs_total"]
for idx, col in enumerate(vital_plot_cols):
    ax = axes[idx // 3][idx % 3]
    if col not in train_raw.columns:
        continue
    for acuity in [1, 2, 3, 4, 5]:
        data = train_raw[train_raw["triage_acuity"] == acuity][col].dropna()
        if len(data) > 0:
            ax.hist(data, bins=40, alpha=0.4, label=f"ESI-{acuity}",
                    color=PALETTE[acuity - 1], density=True)
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.legend(fontsize=8)
    ax.set_xlabel(col)
plt.suptitle("Vital Sign Distributions by Triage Acuity", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig3_vitals_by_acuity.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -- 6d: Chief complaint system distribution --
if "chief_complaint_system" in train_raw.columns:
    fig, ax = plt.subplots(figsize=(14, 8))
    ct = pd.crosstab(train_raw["chief_complaint_system"],
                     train_raw["triage_acuity"], normalize="index")
    ct.plot(kind="barh", stacked=True, ax=ax, color=PALETTE, edgecolor="white")
    ax.set_xlabel("Proportion", fontweight="bold")
    ax.set_title("ESI Acuity Distribution by Chief Complaint System", fontweight="bold")
    ax.legend(title="ESI", labels=[f"ESI-{i}" for i in range(1, 6)])
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig4_complaint_system.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# -- 6e: Comorbidity analysis --
hx_cols = [c for c in history.columns if c.startswith("hx_")]
train_with_hx = train_raw.merge(history, on="patient_id", how="left")
train_with_hx["total_hx"] = train_with_hx[hx_cols].sum(axis=1)
print("Mean comorbidity count by acuity:")
for acuity, val in train_with_hx.groupby("triage_acuity")["total_hx"].mean().items():
    print(f"  ESI-{acuity}: {val:.2f}")

fig, ax = plt.subplots(figsize=(14, 8))
hx_by_acuity = train_with_hx.groupby("triage_acuity")[hx_cols].mean()
hx_by_acuity.T.plot(kind="barh", ax=ax, color=PALETTE, edgecolor="white")
ax.set_xlabel("Prevalence Rate", fontweight="bold")
ax.set_title("Comorbidity Prevalence by ESI Acuity Level", fontweight="bold")
ax.legend(title="ESI", labels=[f"ESI-{i}" for i in range(1, 6)])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig5_comorbidities.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -- 6f: Arrival patterns --
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
hourly = pd.crosstab(train_raw["arrival_hour"], train_raw["triage_acuity"], normalize="index")
hourly.plot(kind="area", stacked=True, ax=axes[0], color=PALETTE, alpha=0.8)
axes[0].set_title("Acuity Mix by Arrival Hour", fontweight="bold")
axes[0].set_xlabel("Hour of Day", fontweight="bold")
axes[0].set_ylabel("Proportion", fontweight="bold")
axes[0].legend(title="ESI", labels=[f"ESI-{i}" for i in range(1, 6)])

mode_acuity = pd.crosstab(train_raw["arrival_mode"], train_raw["triage_acuity"], normalize="index")
mode_acuity.plot(kind="barh", stacked=True, ax=axes[1], color=PALETTE, edgecolor="white")
axes[1].set_title("Acuity Mix by Arrival Mode", fontweight="bold")
axes[1].set_xlabel("Proportion", fontweight="bold")
axes[1].legend(title="ESI", labels=[f"ESI-{i}" for i in range(1, 6)])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig6_arrival_patterns.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Feature Engineering

The pipeline produces 153 engineered columns and uses 146 modeling features after
dropping IDs, raw text, and outcome-derived columns (`disposition`, `ed_los_hours`,
`site_id`, `triage_nurse_id`).

Feature groups:

- **Raw physiology and intake context**: BP, HR, RR, temperature, SpO2, GCS, pain score,
  age, arrival mode, prior utilization.
- **Derived clinical features**: shock index, MAP, pulse pressure, hemodynamic /
  temperature / respiratory abnormality flags, age-adjusted HR abnormality, critical
  flag counts.
- **Informative missingness**: per-vital missing flags, documentation completeness,
  total missing-vital count.
- **Complaint NLP heuristics**: complaint length, keyword acuity counts, targeted
  high-risk complaint flags.
- **Comorbidity composites**: cardiovascular, metabolic, respiratory, mental health,
  immunocompromised burden summaries.
- **Temporal patterns**: cyclical hour/day/month encodings plus night/evening/weekend
  flags.


In [ ]:
train_df = engineer_features(train_raw.copy(), complaints, history, is_train=True)
test_df = engineer_features(test_raw.copy(), complaints, history, is_train=False)

category_maps = build_category_maps(train_df)
train_df = encode_categoricals(train_df, category_maps)
test_df = encode_categoricals(test_df, category_maps)

feature_cols = get_feature_columns(train_df)
missing_in_test = [c for c in feature_cols if c not in test_df.columns]
if missing_in_test:
    print(f"[!] Adding missing columns to test set: {missing_in_test}")
    for c in missing_in_test:
        test_df[c] = 0
feature_cols = [c for c in feature_cols if c in train_df.columns and c in test_df.columns]

y = train_df["triage_acuity"].values
print(f"\nFinal feature count: {len(feature_cols)}")
print(f"Training samples: {len(train_df)}")
print(f"Test samples:     {len(test_df)}")

groups = {
    "Vital signs (raw)": [c for c in feature_cols if c in [
        "systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate",
        "temperature_c", "spo2", "gcs_total", "pain_score", "weight_kg", "height_cm"]],
    "Clinical derived": [c for c in feature_cols if any(c.startswith(p) for p in [
        "shock_", "map_", "pulse_pressure", "spo2_", "gcs_", "hypo", "hyper",
        "tachy", "brady", "fever", "pain_", "pp_", "hr_age", "n_critical"])],
    "Missingness": [c for c in feature_cols if "missing" in c or c == "doc_completeness"],
    "NLP / Chief complaint": [c for c in feature_cols if c.startswith("cc_") or (
        c.startswith("n_") and "keyword" in c) or c in [
        "complaint_length", "complaint_word_count", "keyword_acuity_signal"]],
    "Comorbidity": [c for c in feature_cols if c.startswith("hx_") or c == "total_hx_flags"],
    "Temporal": [c for c in feature_cols if any(c.startswith(p) for p in [
        "hour_", "day_", "month_", "is_night", "is_evening", "is_weekend"])],
    "Demographics": [c for c in feature_cols if c in [
        "age", "sex", "arrival_mode", "arrival_hour", "arrival_day", "arrival_month"]],
}
print("\nFeature groups:")
for name, cols in groups.items():
    if cols:
        print(f"  {name:<25} {len(cols):>3} features")


## 8. Modeling Pipeline

Three base learners with 5-fold stratified CV (all evaluated on honest out-of-fold
predictions) feed a multinomial logistic meta-learner. The stacked ensemble is then
passed through an entropy-triggered **conservative shift** that nudges uncertain
predictions one level toward higher acuity.

| Stage | Model | Notes |
|---|---|---|
| Baseline | LightGBM | Standard cross-entropy, no class weights |
| Safety-weighted | LightGBM | Higher sample weights for ESI-1/2/5 |
| Safety-weighted | XGBoost | Diversity for the stacking meta-learner |
| Stacking | Multinomial logistic | Own 5-fold OOF evaluation |
| Post-processing | Conservative shift | Entropy threshold swept on OOF meta probs |


In [ ]:
# -- 8a: Baseline LightGBM (standard cross-entropy, no safety weighting) --
print("-- 8a: Baseline LightGBM --")
baseline_models, baseline_oof, baseline_preds = train_lgbm_cv(
    train_df, y, feature_cols,
    n_folds=5,
    params=LGBM_BASE_PARAMS.copy(),
    use_sample_weights=False,
    model_name="Baseline LightGBM",
)
baseline_report = full_safety_report(y, baseline_preds, model_name="Baseline LightGBM")
print_safety_report(baseline_report)


In [ ]:
# -- 8b: Safety-Weighted LightGBM --
print("-- 8b: Safety-Weighted LightGBM --")
safety_lgbm_models, safety_lgbm_oof, safety_lgbm_preds = train_lgbm_cv(
    train_df, y, feature_cols,
    n_folds=5,
    params=LGBM_BASE_PARAMS.copy(),
    use_sample_weights=True,
    model_name="Safety-Weighted LightGBM",
)
safety_lgbm_report = full_safety_report(y, safety_lgbm_preds, model_name="Safety-Weighted LightGBM")
print_safety_report(safety_lgbm_report)


In [ ]:
# -- 8c: XGBoost (diversity for ensemble) --
print("-- 8c: Safety-Weighted XGBoost --")
xgb_models, xgb_oof, xgb_preds = train_xgb_cv(
    train_df, y, feature_cols,
    n_folds=5,
    use_sample_weights=True,
    model_name="Safety-Weighted XGBoost",
)
xgb_report = full_safety_report(y, xgb_preds, model_name="Safety-Weighted XGBoost")
print_safety_report(xgb_report)


In [ ]:
# -- 8d: Stacking meta-learner --
print("-- 8d: Stacking meta-learner --")
oof_dict = {
    "lgbm_baseline": baseline_oof,
    "lgbm_safety":   safety_lgbm_oof,
    "xgb_safety":    xgb_oof,
}
meta_model, meta_scaler, meta_oof_proba, meta_oof_labels = train_stacking_meta(oof_dict, y)
stacked_report = full_safety_report(y, meta_oof_labels, model_name="Stacked Ensemble")
print_safety_report(stacked_report)


In [ ]:
# -- 8e: Uncertainty-aware conservative shift --
print("-- 8e: Conservative shift threshold sweep --")
best_threshold = 0.7
best_cwe = float("inf")
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    shifted = conservative_shift(meta_oof_proba, entropy_threshold=threshold)
    cwe = cost_weighted_error(y, shifted)
    utr = undertriage_rate(y, shifted)
    acc = (shifted == y).mean()
    print(f"  threshold={threshold:.1f}  Acc={acc:.4f}  UTR={utr:.4f}  CWE={cwe:.4f}")
    if cwe < best_cwe:
        best_cwe = cwe
        best_threshold = threshold

print(f"\n  -> Best entropy threshold: {best_threshold}")
final_oof_preds = conservative_shift(meta_oof_proba, entropy_threshold=best_threshold)
final_report = full_safety_report(y, final_oof_preds, model_name="SafeTriageNet (Final)")
print_safety_report(final_report)


## 9. Evaluation and Clinical Safety Analysis

We compare five configurations side-by-side on accuracy, macro-F1, under-triage rate,
severe under-triage rate, over-triage rate, and cost-weighted error. The safety-aware
models trade a negligible amount of raw accuracy for a materially lower under-triage
rate — which is what we care about clinically.


In [ ]:
all_reports = [baseline_report, safety_lgbm_report, xgb_report, stacked_report, final_report]
comparison_df = pd.DataFrame(all_reports)
comparison_cols = [
    "model_name", "accuracy", "macro_f1", "weighted_f1",
    "quadratic_kappa", "undertriage_rate", "severe_undertriage_rate",
    "overtriage_rate", "cost_weighted_error", "safety_adjusted_kappa",
]
print(comparison_df[comparison_cols].to_string(index=False))
comparison_df[comparison_cols].to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


In [ ]:
# -- 9a: Confusion matrix comparison --
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for ax, (preds, name) in zip(axes, [
    (baseline_preds, "Baseline LightGBM"),
    (meta_oof_labels, "Stacked Ensemble"),
    (final_oof_preds, "SafeTriageNet (Final)"),
]):
    cm = confusion_matrix(y, preds, labels=[1, 2, 3, 4, 5])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax,
        xticklabels=[f"ESI-{i}" for i in range(1, 6)],
        yticklabels=[f"ESI-{i}" for i in range(1, 6)],
        linewidths=1, linecolor="white",
        cbar_kws={"label": "Proportion"},
    )
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("Actual", fontweight="bold")
    ax.set_title(name, fontweight="bold")
plt.suptitle("Confusion Matrices: Baseline vs Stacked vs Safety-Aware",
             fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig7_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -- 9b: Safety / accuracy tradeoff --
fig, ax = plt.subplots(figsize=(10, 7))
for report, marker, size in zip(
    all_reports,
    ["o", "s", "D", "^", "*"],
    [100, 100, 100, 120, 200],
):
    ax.scatter(
        report["accuracy"], report["undertriage_rate"],
        s=size, marker=marker, label=report["model_name"],
        edgecolors="black", linewidth=1, zorder=5,
    )
ax.axhline(y=0.05, color="red", linestyle="--", alpha=0.7, label="ACS 5% UTR Threshold")
ax.set_xlabel("Accuracy", fontweight="bold", fontsize=13)
ax.set_ylabel("Under-Triage Rate (ESI 1-2 -> ESI 3-5)", fontweight="bold", fontsize=13)
ax.set_title("Safety-Accuracy Tradeoff Frontier\nLower Under-Triage Rate = Safer",
             fontweight="bold", fontsize=14)
ax.legend(loc="upper right", fontsize=10)
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig8_safety_accuracy_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -- 9c: Feature importance --
importance_df = get_feature_importance(safety_lgbm_models, feature_cols, importance_type="gain")
print("Top 30 features by mean gain:")
print(importance_df.head(30).to_string(index=False))

top_n = 30
top_features = importance_df.head(top_n).reset_index(drop=True)


def _feature_color(name: str) -> str:
    if "missing" in name or name == "doc_completeness":
        return "#e74c3c"       # red  -- missingness
    if (name.startswith("cc_")
            or "keyword" in name
            or name in {"complaint_length", "complaint_word_count", "keyword_acuity_signal"}):
        return "#2ecc71"       # green -- NLP
    if name.startswith("hx_"):
        return "#9b59b6"       # purple -- comorbidity
    return "#3498db"           # blue  -- other structured


feature_colors = [_feature_color(f) for f in top_features["feature"]]
y_positions = list(range(top_n - 1, -1, -1))  # top feature first

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(
    y_positions, top_features["mean_importance"].values,
    xerr=top_features["std_importance"].values,
    color=feature_colors, edgecolor="white", alpha=0.9, capsize=3,
)
ax.set_yticks(y_positions)
ax.set_yticklabels(top_features["feature"].values, fontsize=9)
ax.set_xlabel("Mean Gain (Feature Importance)", fontweight="bold")
ax.set_title(
    "Top 30 Features -- Safety-Weighted LightGBM\n"
    "(red: missingness, green: NLP/complaint, purple: comorbidity, blue: other)",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig9_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)


In [ ]:
# -- 9d: Fairness audit by sex and age group --
print("\nFairness audit:")
for group_col, source_series in [
    ("Sex", train_raw["sex"]),
    ("Age Group", train_raw["age_group"]),
]:
    print(f"\n  Performance by {group_col}:")
    for group in sorted(source_series.dropna().unique()):
        mask = source_series.values == group
        if mask.sum() < 50:
            continue
        group_true = y[mask]
        group_pred = final_oof_preds[mask]
        acc = (group_true == group_pred).mean()
        utr_ = undertriage_rate(group_true, group_pred)
        cwe_ = cost_weighted_error(group_true, group_pred)
        print(f"    {str(group):<20} n={mask.sum():>6,}  "
              f"Acc={acc:.4f}  UTR={utr_:.4f}  CWE={cwe_:.4f}")


## 10. SHAP Explainability

Global (all-class) and ESI-1-specific SHAP summaries for the safety-weighted LightGBM.
SHAP is optional — if the package is unavailable, the block is skipped without failing
the rest of the notebook.


In [ ]:
try:
    import shap

    n_shap_samples = min(2000, len(train_df))
    rng_shap = np.random.default_rng(SEED)
    shap_idx = rng_shap.choice(len(train_df), size=n_shap_samples, replace=False)
    X_shap = train_df.iloc[shap_idx][feature_cols]

    explainer = shap.TreeExplainer(safety_lgbm_models[0])
    shap_values = explainer.shap_values(X_shap)

    if isinstance(shap_values, list):
        shap_per_class = shap_values
        mean_abs_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_per_class = [shap_values[:, :, c] for c in range(shap_values.shape[2])]
        mean_abs_shap = np.mean(np.abs(shap_values), axis=2)
    else:
        shap_per_class = None
        mean_abs_shap = np.abs(shap_values)

    fig, ax = plt.subplots(figsize=(14, 10))
    feature_shap_means = mean_abs_shap.mean(axis=0)
    top_idx = np.argsort(feature_shap_means)[-20:]
    shap_df = pd.DataFrame({
        "feature": [feature_cols[i] for i in top_idx],
        "mean_abs_shap": feature_shap_means[top_idx],
    }).sort_values("mean_abs_shap", ascending=True)
    ax.barh(range(len(shap_df)), shap_df["mean_abs_shap"].values,
            color="steelblue", edgecolor="white", alpha=0.9)
    ax.set_yticks(range(len(shap_df)))
    ax.set_yticklabels(shap_df["feature"].values)
    ax.set_xlabel("Mean |SHAP Value|", fontweight="bold")
    ax.set_title("SHAP Feature Importance -- Top 20 Features\n"
                 "(Mean absolute SHAP across all ESI classes)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig10_shap_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

    if shap_per_class is not None and len(shap_per_class) >= 1:
        fig, ax = plt.subplots(figsize=(14, 10))
        esi1_shap = np.abs(shap_per_class[0]).mean(axis=0)
        top_esi1 = np.argsort(esi1_shap)[-15:]
        esi1_df = pd.DataFrame({
            "feature": [feature_cols[i] for i in top_esi1],
            "mean_abs_shap": esi1_shap[top_esi1],
        }).sort_values("mean_abs_shap", ascending=True)
        ax.barh(range(len(esi1_df)), esi1_df["mean_abs_shap"].values,
                color="#e74c3c", edgecolor="white", alpha=0.9)
        ax.set_yticks(range(len(esi1_df)))
        ax.set_yticklabels(esi1_df["feature"].values)
        ax.set_xlabel("Mean |SHAP Value|", fontweight="bold")
        ax.set_title("SHAP Feature Importance -- ESI-1 (Resuscitation)\n"
                     "What drives the highest-acuity predictions?", fontweight="bold")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "fig11_shap_esi1.png", dpi=150, bbox_inches="tight")
        plt.show()
except Exception as exc:
    print(f"SHAP analysis skipped: {exc}")


## 11. Clinical Insights and Limitations

Three findings stand out:

1. **Informative missingness is real but partial.** Four of six core vitals show
   significant acuity-linked missingness. Documentation completeness is one of the
   strongest features. `heart_rate` and `spo2` are not missing in this dataset — we
   therefore describe missingness as a useful but partial signal, not a universal rule.
2. **Safety vs accuracy trade-off is what the project buys.** Overall accuracy is
   essentially flat across all five model configurations. What changes is *which*
   mistakes are made — the final system reduces under-triage of ESI-1/2 patients
   relative to the accuracy-optimized baseline.
3. **A broad multimodal feature set beats a complicated modeling stack.** GCS, NEWS2,
   pain score, and SpO2 dominate, followed by complaint-text heuristics and comorbidity
   composites. We did not need a heavyweight language model to benefit from text.

**Limitations.**

1. Synthetic data. Real ED data is noisier and more institution-specific.
2. Lightweight text features only. Real triage notes contain abbreviations,
   misspellings, and clinical shorthand this pipeline does not model.
3. The conservative-shift threshold is selected from the same out-of-fold predictions
   used for evaluation — acceptable for a competition notebook, but not a fully nested
   estimate.
4. No probability calibration (Platt / isotonic). Clinical decision support requires
   calibrated confidence scores before any real deployment.
5. No prospective validation. Any downstream clinical use would require testing on real
   ED data and, in practice, a prospective study against current nurse triage.


## 12. Generate Submission

Predictions come from the full stacking pipeline (averaged base-model predictions
over the five folds, then meta-learner, then conservative shift at the selected
entropy threshold). The submission file is written to the working directory.


In [ ]:
print("Generating test predictions...")
models_dict = {
    "lgbm_baseline": baseline_models,
    "lgbm_safety":   safety_lgbm_models,
    "xgb_safety":    xgb_models,
}
test_proba, _ = predict_stacked(
    models_dict,
    test_df,
    feature_cols,
    meta_model,
    meta_scaler,
)
test_final_preds = conservative_shift(test_proba, entropy_threshold=best_threshold)

submission = pd.DataFrame({
    "patient_id": test_df["patient_id"],
    "triage_acuity": test_final_preds.astype(int),
})

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"\nSubmission saved to: {submission_path}")
print(f"Submission shape:    {submission.shape}")
print("\nPrediction distribution:")
print(submission["triage_acuity"].value_counts().sort_index().to_string())

assert list(submission.columns) == list(sample_sub.columns), "Submission format mismatch!"
assert len(submission) == len(sample_sub), "Submission length mismatch!"
print("\n[OK] Submission format verified.")

print(f"\nFinal SafeTriageNet metrics (out-of-fold):")
print(f"  Accuracy:      {final_report['accuracy']:.4f}")
print(f"  Macro F1:      {final_report['macro_f1']:.4f}")
print(f"  Under-triage:  {final_report['undertriage_rate']:.4f}")
print(f"  CWE:           {final_report['cost_weighted_error']:.4f}")
